# Data — Combine Persona Subsets

Merges the per-persona subsets into the pooled train/test splits used by the centralised runs and by evaluation.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [2]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U pandas pyarrow

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )

SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"
ALL_SUBSET_NAME = "all_subset"
ALL_SUBSET_DIR = SUBSETS_DIR / ALL_SUBSET_NAME
ALL_SUBSET_DIR.mkdir(parents=True, exist_ok=True)
print("Reading subsets from:", SUBSETS_DIR.resolve())
print("Merged data will be saved to:", ALL_SUBSET_DIR.resolve())

Mounted at /content/drive
Reading subsets from: /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets
Merged data will be saved to: /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/all_subset


In [3]:
import json

import pandas as pd


def load_manifest(subsets_dir=SUBSETS_DIR):
    with open(Path(subsets_dir) / "manifest.json", "r", encoding="utf-8") as f:
        return json.load(f)


def list_subsets(subsets_dir=SUBSETS_DIR):
    """Names of the individual subsets (subset_0, subset_1, ...), excluding all_subset."""
    return sorted(n for n in load_manifest(subsets_dir)["subsets"].keys() if n != ALL_SUBSET_NAME)


def load_subset(name, subsets_dir=SUBSETS_DIR):
    """Return (persona_ids, train_df, val_df) for a saved subset."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"No such subset folder: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    sub_train = pd.read_parquet(subset_dir / "train.parquet")
    sub_val = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), sub_train, sub_val


subset_names = list_subsets()
print("Subsets found:", subset_names)

train_parts, val_parts, all_persona_ids = [], [], []
for name in subset_names:
    ids, tr, va = load_subset(name)
    tr = tr.copy(); va = va.copy()
    tr["subset"] = name
    va["subset"] = name
    train_parts.append(tr)
    val_parts.append(va)
    all_persona_ids.extend(ids)
    print(f"  {name}: {len(ids)} personas | train {len(tr):,} rows | val {len(va):,} rows")

all_persona_ids = sorted(set(all_persona_ids))
print(f"\nTotal personas across subsets: {len(all_persona_ids)}")

Subsets found: ['subset_0', 'subset_1', 'subset_2']
  subset_0: 50 personas | train 1,267 rows | val 230 rows
  subset_1: 50 personas | train 1,302 rows | val 240 rows
  subset_2: 50 personas | train 1,301 rows | val 244 rows

Total personas across subsets: 150


In [4]:
all_train_df = pd.concat(train_parts, ignore_index=True)
all_val_df = pd.concat(val_parts, ignore_index=True)

print(f"all_train_df: {len(all_train_df):,} rows | {all_train_df['persona_id'].nunique()} personas")
print(f"all_val_df:   {len(all_val_df):,} rows | {all_val_df['persona_id'].nunique()} personas")
print("\nColumns:", list(all_train_df.columns))

all_train_df: 3,870 rows | 150 personas
all_val_df:   714 rows | 150 personas

Columns: ['persona_id', 'chat_history_32k_link', 'chat_history_128k_link', 'raw_persona_file', 'short_persona', 'expanded_persona', 'user_query', 'correct_answer', 'incorrect_answers', 'topic_query', 'preference', 'topic_preference', 'conversation_scenario', 'pref_type', 'related_conversation_snippet', 'who', 'updated', 'prev_pref', 'sensitive_info', 'total_tokens_in_chat_history_32k', 'total_tokens_in_chat_history_128k', 'distance_from_related_snippet_to_query_32k', 'distance_from_related_snippet_to_query_128k', 'num_persona_relevant_tokens_32k', 'num_persona_irrelevant_tokens_32k', 'num_persona_relevant_tokens_128k', 'num_persona_irrelevant_tokens_128k', 'negative_conversation_snippet', 'negative_snippet_distance_turns', 'negative_snippet_similarity', 'negative_snippet_source', 'subset']


In [5]:
all_train_df.to_parquet(ALL_SUBSET_DIR / "train.parquet", index=False)
all_val_df.to_parquet(ALL_SUBSET_DIR / "val.parquet", index=False)
with open(ALL_SUBSET_DIR / "personas.json", "w", encoding="utf-8") as f:
    json.dump([int(p) for p in all_persona_ids], f, indent=2)

print("Saved merged train ->", (ALL_SUBSET_DIR / "train.parquet").resolve())
print("Saved merged val   ->", (ALL_SUBSET_DIR / "val.parquet").resolve())
print("Saved personas.json->", (ALL_SUBSET_DIR / "personas.json").resolve())

_ids, _tr, _va = load_subset(ALL_SUBSET_NAME)
print(f"\nRead back all_subset: {len(_ids)} personas | train {len(_tr):,} rows | val {len(_va):,} rows")

Saved merged train -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/all_subset/train.parquet
Saved merged val   -> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/all_subset/val.parquet
Saved personas.json-> /content/drive/MyDrive/privacy_perserving_pllm/v2_personamem_persona_subsets/all_subset/personas.json

Read back all_subset: 150 personas | train 3,870 rows | val 714 rows


In [8]:
import ast
from typing import Any


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def parse_turns(raw):
    """Parse a snippet (JSON string / list) into a list of {role, content} turns."""
    if isinstance(raw, list):
        return raw
    if raw is None or isinstance(raw, float):
        return []
    try:
        val = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        try:
            val = ast.literal_eval(raw)
        except Exception:
            return []
    return val if isinstance(val, list) else []


def turns_text(turns):
    return "\n".join(
        f"  [{t.get('role', '')}] {t.get('content', '')}" for t in turns if isinstance(t, dict)
    )


def show_snippet(raw, max_chars=800):
    turns = parse_turns(raw)
    text = turns_text(turns) if turns else str(raw)
    if max_chars is None:
        return text
    return text[:max_chars] + (" ..." if len(text) > max_chars else "")


N_EXAMPLES = 3
sample = all_train_df.sample(n=min(N_EXAMPLES, len(all_train_df)), random_state=42)

for i, (_, row) in enumerate(sample.iterrows(), 1):
    print("=" * 100)
    print(f"EXAMPLE {i} | subset={row.get('subset')} | persona_id={row.get('persona_id')}")
    print("=" * 100)
    print("\nQUERY:")
    print(" ", parse_user_query(row.get("user_query")))
    print("\nRELATED (POSITIVE) SNIPPET:")
    print(show_snippet(row.get("related_conversation_snippet")))
    print("\nNEGATIVE SNIPPET:")
    print(show_snippet(row.get("negative_conversation_snippet")))
    print()

EXAMPLE 1 | subset=subset_1 | persona_id=618

QUERY:
  Hi there, I’m in the midst of organizing a multi-city executive retreat for senior leaders within our network. The main segments will be held in Nairobi followed by a coastal getaway in Mombasa. I need creative guidance on drafting digital invitations that balance exclusivity and a personable executive tone. Also, I’m looking for suggestions on streamlining RSVPs and follow-ups via email. For context, I typically send these invitations from my official email (nia.kamau.private@emailkenya.co.ke). Could you help me design an approach that ensures professionalism, confidentiality, and ease-of-response?

RELATED (POSITIVE) SNIPPET:
  [user] Hi, I'm drafting a professional email to confirm details regarding an upcoming investor conference at our Nairobi office. I pasted my draft email below. I want the language to be clear, concise, and polished. Also, please don’t remove any of my contact details as they are necessary for follow-up. He

In [12]:
import numpy as np
from transformers import AutoTokenizer

try:
    import transformers
except ImportError:
    !pip -q install transformers sentencepiece
    import transformers

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")

def get_token_length_combined(query_raw, snippet_raw, tokenizer):
    """Combines query and snippet text, then tokenizes and returns length."""
    query_text = parse_user_query(query_raw)
    snippet_text = show_snippet(snippet_raw, max_chars=None)
    combined_text = f"QUERY: {query_text}\nSNIPPET: {snippet_text}"
    return len(tokenizer.encode(combined_text))

def get_token_length_single(text_raw, tokenizer):
    """Tokenizes a single text input and returns its length."""
    text = parse_user_query(text_raw)
    return len(tokenizer.encode(text))

def get_second_largest(token_lengths_list):
    """Returns the second largest element in a list, or N/A if not enough elements."""
    if len(token_lengths_list) < 2:
        return "N/A (less than 2 elements)"
    sorted_list = sorted(token_lengths_list, reverse=True)
    return sorted_list[1]

neg_snippet_query_lengths = []
rel_snippet_query_lengths = []
correct_answer_lengths = []
incorrect_answer_lengths = []

prompt = """You are a personalised assistant. Use the conversation snippet to find
        information or connections relevant to the question, then provide the answer."""

for _, row in all_train_df.iterrows():
    neg_snippet_query_lengths.append(
        get_token_length_combined(prompt + row.get("user_query"), row.get("negative_conversation_snippet"), tokenizer)
    )

    rel_snippet_query_lengths.append(
        get_token_length_combined(prompt + row.get("user_query"), row.get("related_conversation_snippet"), tokenizer)
    )

    correct_answer_lengths.append(
        get_token_length_single(row.get("correct_answer"), tokenizer)
    )

    incorrect_answer_lengths.append(
        get_token_length_single(row.get("incorrect_answers"), tokenizer)
    )

print("--- Negative Snippet + Query Token Lengths ---")
print(f"Average: {np.mean(neg_snippet_query_lengths):.2f}")
print(f"Max: {np.max(neg_snippet_query_lengths)}")
print(f"Min: {np.min(neg_snippet_query_lengths)}")
print(f"Second Largest: {get_second_largest(neg_snippet_query_lengths)}")

print("\n--- Relevant Snippet + Query Token Lengths ---")
print(f"Average: {np.mean(rel_snippet_query_lengths):.2f}")
print(f"Max: {np.max(rel_snippet_query_lengths)}")
print(f"Min: {np.min(rel_snippet_query_lengths)}")
print(f"Second Largest: {get_second_largest(rel_snippet_query_lengths)}")

print("\n--- Correct Answer Token Lengths ---")
print(f"Average: {np.mean(correct_answer_lengths):.2f}")
print(f"Max: {np.max(correct_answer_lengths)}")
print(f"Min: {np.min(correct_answer_lengths)}")
print(f"Second Largest: {get_second_largest(correct_answer_lengths)}")

print("\n--- Incorrect Answer Token Lengths ---")
print(f"Average: {np.mean(incorrect_answer_lengths):.2f}")
print(f"Max: {np.max(incorrect_answer_lengths)}")
print(f"Min: {np.min(incorrect_answer_lengths)}")
print(f"Second Largest: {get_second_largest(incorrect_answer_lengths)}")

--- Negative Snippet + Query Token Lengths ---
Average: 339.45
Max: 2073
Min: 70
Second Largest: 1830

--- Relevant Snippet + Query Token Lengths ---
Average: 467.40
Max: 1349
Min: 117
Second Largest: 1221

--- Correct Answer Token Lengths ---
Average: 66.82
Max: 120
Min: 8
Second Largest: 119

--- Incorrect Answer Token Lengths ---
Average: 193.05
Max: 345
Min: 1
Second Largest: 315


In [11]:
row.get("incorrect_answer")